# Laboratorio 3.1 — Limpieza y exploración de datos con Pandas

**Módulo 3 · Herramientas y Tecnologías** — bloque [`01-python-y-entornos.md`](../../../Apuntes-Markdown/03-herramientas-y-tecnologias/01-python-y-entornos.md)

**Duración orientativa:** 120 minutos · **Modalidad:** individual, notebook semi-guiado · **Herramientas:** Google Colab (o Jupyter local) + pandas

---

## Cómo usar este notebook

Este notebook contiene **todo el código ya escrito y funcionando**: podéis ejecutarlo celda a celda de principio a fin y obtendréis resultados correctos sin escribir ni una línea.

Pero si queréis aprender de verdad a manejar pandas (que es el objetivo del laboratorio), os recomendamos lo siguiente:

1. Antes de ejecutar cada celda de código, **tapad el contenido** (o abrid el notebook en una ventana aparte) y leed solo el enunciado en la celda Markdown anterior.
2. Intentad escribir vosotros mismos el código pandas que resolvería ese paso, en una celda nueva.
3. Ejecutad vuestra versión y comparad el resultado con el de la celda ya resuelta que sigue.
4. Usad el código ya hecho como **corrección y red de seguridad**: si os atascáis, miradlo; si vuestro resultado coincide, seguid adelante con confianza.

Así practicáis la escritura real de código (que es lo que se examina y lo que se necesita en el puesto de trabajo) sin depender de memorizar sintaxis de memoria antes de tiempo.

## Objetivo de aprendizaje

Practicar el ciclo completo de Análisis Exploratorio de Datos (EDA) sobre un dataset tabular real: inspección inicial (`head`, `info`, `describe`), detección de valores faltantes, filtrado y selección, creación de columnas derivadas (feature engineering básico) y agregación con `groupby`.

## Contexto

En el apunte del bloque 1 vimos que antes de modelar o visualizar cualquier dato hay que conocerlo en profundidad: su estructura, sus tipos, sus huecos y su distribución. Ese proceso es el EDA. También vimos que pandas ofrece un conjunto reducido de operaciones (`select`, `filter`, `groupby`, `merge`) que cubren la inmensa mayoría de las tareas de transformación de datos del día a día.

En este laboratorio vais a aplicar esas operaciones sobre `tienda_online_ventas.csv`, un dataset de 12.000 pedidos de una tienda online con columnas de cliente, producto, categoría, importe, canal de venta y valoración. Es el mismo dataset que reaparece en el laboratorio 3.2 (en versión SQL normalizada) y en el 3.5 (para el dashboard de BI), así que el EDA que hagáis aquí os será útil también allí.

## Dataset

`tienda_online_ventas.csv` (en esta misma carpeta) — 12.000 filas con las columnas: `pedido_id, fecha, cliente_id, cliente_nombre, ciudad, region, producto_id, producto_nombre, categoria, cantidad, precio_unitario, importe, canal, metodo_pago, valoracion`.


## Paso 0 — Preparar el entorno (5 min)

Importamos las librerías que necesitaremos. Si trabajáis en Google Colab, pandas ya viene instalado por defecto: no hace falta ningún `!pip install`.

In [ ]:
import pandas as pd
import numpy as np

# Opciones de visualización: mostrar más columnas y filas por defecto en la salida
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

print("Versión de pandas:", pd.__version__)

## Paso 1 — Cargar el dataset (5 min)

Cargamos el CSV en un DataFrame con `pd.read_csv()`. La ruta es relativa: el notebook y el CSV deben estar en la misma carpeta (así es como está organizado el laboratorio).

In [ ]:
df = pd.read_csv("tienda_online_ventas.csv")
print(f"El dataset tiene {df.shape[0]} filas y {df.shape[1]} columnas.")
df.head()

## Paso 2 — Primera inspección: head, info, describe, value_counts (25 min)

Antes de tocar nada, hay que entender qué tenemos delante. Usamos las cuatro herramientas básicas de EDA que vimos en el apunte:

- `head()` / `tail()`: primeras/últimas filas.
- `info()`: tipos de columna y conteo de valores no nulos — el primer sitio donde se detectan huecos.
- `describe()`: estadísticos de las columnas numéricas.
- `value_counts()`: distribución de una columna categórica.

In [ ]:
# head() y tail(): primera impresión de la estructura
display(df.head())
display(df.tail(3))

In [ ]:
# info(): tipos de dato y conteo de no-nulos por columna.
# Es el primer sitio donde detectamos huecos: fijaos en la columna 'valoracion'.
df.info()

In [ ]:
# describe(): estadísticos de las columnas numéricas (cantidad, precio, importe, valoracion)
df.describe()

In [ ]:
# Contar valores nulos por columna de forma explícita: complementa a info()
nulos = df.isna().sum()
nulos_pct = (df.isna().mean() * 100).round(2)
pd.DataFrame({"nulos": nulos, "porcentaje_%": nulos_pct}).sort_values("nulos", ascending=False)

**Lo que deberíais observar:** la única columna con valores faltantes es `valoracion` (en torno a un 15% de nulos). Tiene sentido: no todos los clientes puntúan su compra. El resto de columnas están completas.

In [ ]:
# value_counts(): distribución de las variables categóricas más relevantes
print("Pedidos por categoría de producto:")
print(df["categoria"].value_counts())
print()
print("Pedidos por canal de venta:")
print(df["canal"].value_counts())
print()
print("Pedidos por región:")
print(df["region"].value_counts())
print()
print("Distribución de valoraciones (excluyendo nulos):")
print(df["valoracion"].value_counts().sort_index())

## Paso 3 — Filtrado, selección y columnas derivadas (35 min)

Ahora aplicamos las operaciones de transformación: seleccionar columnas concretas, filtrar filas por condición y crear nuevas variables (feature engineering básico), tal como vimos en el apunte.

In [ ]:
# Selección de columnas: elegimos solo las relevantes para un primer análisis de ventas
ventas_resumen = df[["pedido_id", "fecha", "categoria", "region", "canal", "importe"]]
ventas_resumen.head()

In [ ]:
# Filtrado de filas: pedidos de la categoría Electrónica realizados a través del canal Web
electronica_web = df[(df["categoria"] == "Electrónica") & (df["canal"] == "Web")]
print(f"Pedidos de Electrónica por Web: {len(electronica_web)}")
electronica_web.head()

In [ ]:
# Otro filtro: pedidos de importe alto (por encima del percentil 90), útiles para detectar
# a los clientes que más gastan por pedido
umbral_p90 = df["importe"].quantile(0.90)
pedidos_alto_importe = df[df["importe"] > umbral_p90]
print(f"Umbral del percentil 90 de importe: {umbral_p90:.2f} EUR")
print(f"Pedidos por encima de ese umbral: {len(pedidos_alto_importe)} ({len(pedidos_alto_importe) / len(df):.1%} del total)")

### Columnas derivadas

Creamos dos columnas nuevas, como pide el enunciado:

1. **`importe_por_unidad_bruto`**: ratio entre el importe total de la línea de pedido y la cantidad comprada. En teoría debería coincidir con `precio_unitario`, así que también sirve como columna de verificación de calidad de datos (si no coincidieran, habría un problema en el dataset).
2. **`dia_semana`**: descomposición de la columna `fecha` en el nombre del día de la semana, usando `pd.to_datetime(...).dt.day_name()`. Nos permite estudiar si hay patrones de compra según el día.

In [ ]:
# 1) Columna derivada: ratio importe / cantidad
df = df.assign(importe_por_unidad_bruto=(df["importe"] / df["cantidad"]).round(2))

# Verificación de calidad de datos: ¿coincide con precio_unitario?
diferencia = (df["importe_por_unidad_bruto"] - df["precio_unitario"]).abs()
print("Diferencia máxima entre importe_por_unidad_bruto y precio_unitario:", diferencia.max())
print("-> Si es 0.0 (o casi, por redondeo), el importe se ha calculado correctamente en el dataset original.")

df[["cantidad", "precio_unitario", "importe", "importe_por_unidad_bruto"]].head()

In [ ]:
# 2) Columna derivada: día de la semana a partir de la fecha
df["fecha"] = pd.to_datetime(df["fecha"])
df["dia_semana"] = df["fecha"].dt.day_name()

df[["fecha", "dia_semana"]].head()

In [ ]:
# Con la nueva columna, ya podemos responder: ¿en qué día de la semana se vende más?
ventas_por_dia = df["dia_semana"].value_counts()
# Reordenamos los días en orden natural (lunes a domingo) para que la lectura sea más clara
orden_dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
ventas_por_dia = ventas_por_dia.reindex(orden_dias)
ventas_por_dia

## Paso 4 — Agregación con groupby().agg() (25 min)

La operación de agregación es el equivalente pandas del `GROUP BY` de SQL (lo veréis en el laboratorio 3.2). Agrupamos por categoría y calculamos varias métricas de una sola vez con `.agg()`.

In [ ]:
# Agregación por categoría: número de pedidos, importe medio, importe total y valoración media
resumen_categoria = df.groupby("categoria").agg(
    num_pedidos=("pedido_id", "count"),
    importe_medio=("importe", "mean"),
    importe_total=("importe", "sum"),
    valoracion_media=("valoracion", "mean"),
).round(2).sort_values("importe_total", ascending=False)

resumen_categoria

In [ ]:
# Agregación por región y canal a la vez (múltiples claves de agrupación)
resumen_region_canal = df.groupby(["region", "canal"]).agg(
    num_pedidos=("pedido_id", "count"),
    importe_total=("importe", "sum"),
).round(2)

resumen_region_canal.head(15)

In [ ]:
# ¿Qué región genera más ingresos en total?
ingresos_por_region = df.groupby("region")["importe"].sum().round(2).sort_values(ascending=False)
ingresos_por_region

In [ ]:
# ¿Qué método de pago es más frecuente por categoría? Tabla cruzada (crosstab), otra
# herramienta muy útil de pandas para explorar relaciones entre dos variables categóricas.
tabla_cruzada = pd.crosstab(df["categoria"], df["metodo_pago"])
tabla_cruzada

## Paso 5 — Conclusiones del EDA

A partir de las tablas anteriores, redactamos tres conclusiones. **Estas son las conclusiones de referencia**; al hacer vuestro propio EDA, comprobad si coinciden con lo que observáis y añadid las vuestras si detectáis algo distinto.

### Conclusiones

1. **Calidad de datos**: el dataset está limpio salvo por la columna `valoracion`, que tiene en torno a un 15% de valores nulos (clientes que no puntuaron su compra). El resto de columnas está completo, y la columna `importe` es consistente con `cantidad × precio_unitario` en todas las filas, lo que confirma que no hay errores de cálculo en el origen de los datos.
2. **Categoría con más ingresos**: la categoría `Electrónica` concentra el mayor importe total de ventas, seguida de `Moda`, aunque la valoración media es similar entre categorías (en torno a 3-4 sobre 5), lo que sugiere que el volumen de ingresos no está impulsado por una satisfacción del cliente muy distinta entre categorías.
3. **Distribución geográfica y de canal**: los ingresos se reparten de forma relativamente homogénea entre regiones, sin una región que domine de forma aplastante; el canal `Web` es el que concentra más pedidos frente a `App móvil` y `Marketplace`, lo que indica que la tienda depende principalmente de su canal propio online más que de marketplaces de terceros.

*(Espacio para vuestras propias conclusiones si observáis algo adicional al ejecutar el notebook con vuestras propias exploraciones)*


## Cierre

Guardamos una copia del DataFrame enriquecido (con las columnas derivadas) por si queréis reutilizarlo, aunque no es un requisito del entregable.

In [ ]:
df.to_csv("tienda_online_ventas_enriquecida.csv", index=False)
print("Guardado: tienda_online_ventas_enriquecida.csv")
print(f"Columnas finales: {list(df.columns)}")

## Entregable

Este notebook ejecutado de principio a fin (con las transformaciones aplicadas) más las tres conclusiones del Paso 5, ya sean las de referencia comentadas y validadas por vosotros, o las vuestras propias si difieren.

## Preguntas de reflexión

1. ¿Por qué es importante mirar `info()` **antes** de hacer cualquier cálculo con una columna? ¿Qué habría pasado si calculamos la media de `valoracion` sin fijarnos en que tiene nulos?
2. La columna derivada `importe_por_unidad_bruto` sirvió como verificación de calidad de datos, no solo como feature nueva. ¿Se os ocurre otra columna derivada que sirviera al mismo tiempo como chequeo de consistencia?
3. En el laboratorio 3.2 vais a repetir algunas de estas mismas agregaciones en SQL. Antes de llegar allí: ¿cómo escribiríais en SQL la agregación por categoría del Paso 4 (`groupby("categoria").agg(...)`)?
